In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, StringType

spark = (
    SparkSession.builder
    .appName("binance-etl")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")     # crypto trades in UTC
    .config("spark.driver.memory", "12g")             # avoid OOM on the write
    .config("spark.sql.shuffle.partitions", "50")
    .getOrCreate()
)

In [12]:
files_path = "../data/klines_processed/features_with_targets"

df = spark.read.parquet(files_path)

In [13]:
import datetime

bounds = df.select(F.min("event_time").alias("t0"), F.max("event_time").alias("t1")).first()
t0, t1 = bounds["t0"], bounds["t1"]
span_s = (t1 - t0).total_seconds()

train_cut = t0 + datetime.timedelta(seconds=span_s * 0.8)
val_cut   = t0 + datetime.timedelta(seconds=span_s * 0.9)

train = df.filter(F.col("event_time") <  train_cut)
val   = df.filter((F.col("event_time") >= train_cut) & (F.col("event_time") < val_cut))
test  = df.filter(F.col("event_time") >= val_cut)

for name, d in [("train", train), ("val", val), ("test", test)]:
    print(f"{name}: {d.count():,} rows")

train: 9,481,160 rows
val: 1,185,140 rows
test: 1,185,160 rows


In [14]:
df.printSchema()

root
 |-- open_time: long (nullable = true)
 |-- open: string (nullable = true)
 |-- high: string (nullable = true)
 |-- low: string (nullable = true)
 |-- close: string (nullable = true)
 |-- volume: string (nullable = true)
 |-- close_time: long (nullable = true)
 |-- quote_volume: string (nullable = true)
 |-- num_trades: long (nullable = true)
 |-- taker_buy_base: string (nullable = true)
 |-- taker_buy_quote: string (nullable = true)
 |-- ignore: string (nullable = true)
 |-- open_time_ms: long (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- close_num: double (nullable = true)
 |-- high_num: double (nullable = true)
 |-- low_num: double (nullable = true)
 |-- volume_num: double (nullable = true)
 |-- taker_buy_base_num: double (nullable = true)
 |-- prev_close_num: double (nullable = true)
 |-- log_return: double (nullable = true)
 |-- sq_return: double (nullable = true)
 |-- rv_15m: double (nullable = true)
 |-- rv_1h: double (nullable = true)
 |-- rv_4h: doub

In [15]:
feature_cols = ["rv_15m", "rv_1h", "rv_4h", "rv_24h",
                "parkinson_15m", "parkinson_1h",
                "volume_zscore", "buy_ratio"]

keep = ["symbol", "event_time"] + feature_cols + ["target_15m", "target_1h"]

train = train.select(*keep)
val   = val.select(*keep)
test  = test.select(*keep)

In [16]:
scale_cols = ["rv_15m", "rv_1h", "rv_4h", "rv_24h",
              "parkinson_15m", "parkinson_1h", "buy_ratio"]   # not volume_zscore

agg_exprs = []
for c in scale_cols:
    agg_exprs += [F.mean(c).alias(f"{c}_mean"), F.stddev(c).alias(f"{c}_std")]
stats = train.groupBy("symbol").agg(*agg_exprs)

def apply_scaling(d):
    d = d.join(stats, on="symbol", how="left")
    for c in scale_cols:
        d = d.withColumn(c, F.try_divide(F.col(c) - F.col(f"{c}_mean"), F.col(f"{c}_std")))
    drops = [f"{c}_mean" for c in scale_cols] + [f"{c}_std" for c in scale_cols]
    return d.drop(*drops)

train_s, val_s, test_s = apply_scaling(train), apply_scaling(val), apply_scaling(test)

stats.write.mode("overwrite").parquet("../data/scaler_stats")

In [17]:
# Drop warm-up nulls (first 24h per symbol has no rv_24h)
feature_cols = scale_cols + ["volume_zscore"]
train_s = train_s.dropna(subset=feature_cols)
val_s   = val_s.dropna(subset=feature_cols)
test_s  = test_s.dropna(subset=feature_cols)

base = "../data/scaled"
train_s.write.mode("overwrite").partitionBy("symbol").parquet(f"{base}/train")
val_s.write.mode("overwrite").partitionBy("symbol").parquet(f"{base}/val")
test_s.write.mode("overwrite").partitionBy("symbol").parquet(f"{base}/test")

print("done")

done


In [18]:
train_s.filter(F.col("symbol") == "BTCUSDT").select(
    F.mean("rv_15m"), F.stddev("rv_15m")
).show()
# expect ~0 and ~1

+--------------------+------------------+
|         avg(rv_15m)|    stddev(rv_15m)|
+--------------------+------------------+
|2.101727360222836...|0.9999999999999727|
+--------------------+------------------+

